# Implementation of Lambert Liu

### Imports and config dicts

In [2]:
import numpy as np 
import polars as pl 
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from numba import njit  
import math 
from collections import namedtuple

In [ ]:
config_dict = {
    # Binning configs
    'fine_bin_mins' : 5,
    'coarse_bin_mins' : 60,

    # Train, burn in, test validation split config
    'train_days' : 7,
    'burn_in_days' : 7,
    'validation_days' : 7,
    'test_days' : 7,
    
    # Min Params
    'mean_min' : 1e-8,
    'var_min' : 1e-8,
    'min_mean_var_diff' : 1e-8,
    
    # Smoothing Strength and w
    'smoothing_strength' : 0.2, 
    'w' : 0.1,
    
    # setting the seed (used in k means) and clustering method
    'seed' : 110,
    'clustering_method' : 'k_means',
    'clustering_matrix_name' : 'u',
    
    # Calibration outputs and degen threshold
    'degen_threshold' : 0.99, 
    'calibration_thresholds' : np.array([0.001, 0.01, 0.05, 0.1, 0.2, 0.5, 0.9], dtype='float64'),
    }

### Loading and saving data 

In [4]:
data_path = '/home/ma/a/alb25/Project/thesis_code/data/processed/intermediate'
input_data = 'train_df'
df = pl.scan_parquet(f'{data_path}/input/{input_data}.parquet')


# Creating functions for storing and reading data
def store_data(data, filename, data_path=data_path, csv=False):
    if isinstance(data, pl.LazyFrame):
        if csv == True:
            data.sink_csv(f'{data_path}/output/{filename}.csv')
        else: 
            data.sink_parquet(f'{data_path}/output/{filename}.parquet')
    elif isinstance(data, pl.DataFrame):
        if csv == True:
            data.write_csv(f'{data_path}/output/{filename}.csv')
        else:
            data.write_parquet(f'{data_path}/output/{filename}.parquet')
    elif isinstance(data, np.ndarray):
        np.save(f'{data_path}/output/{filename}.npy', data)
    else:
        raise TypeError('Function doesnt support this data type')
    

def load_data(filename, data_type, data_path=data_path):
    ''' 
    Arguments:
        filename: 
            the saved file name
        data_type: ['lazy', 'np', 'df']
            the type of data we want to load in
    '''
    if data_type == 'lazy':
        data = pl.scan_parquet(f'{data_path}/output/{filename}.parquet')
    elif data_type == 'np':
        data = np.load(f'{data_path}/output/{filename}.npy')
    elif data_type == 'df':
        data = pl.read_parquet(f'{data_path}/output/{filename}.parquet')
    else:
        raise TypeError('Function doesnt support this data type')
    return data

### Getting helper dictionaries

In [5]:
# Helper functions outputing dicitonaries
def get_bin_metrics(config_dict=config_dict):
    ''' 
    Returns a dictionary containing all information needed about bin length these numbers are needed throughout the pipeline
    '''
    # Check that bin length divides number of days
    assert 24 * 60 % config_dict['fine_bin_mins'] == 0
    assert 24 * 60 % config_dict['coarse_bin_mins'] == 0
    assert config_dict['coarse_bin_mins'] % config_dict['fine_bin_mins'] == 0

    output_dict = {'fine_bins_per_day' : 24 * 60 // config_dict['fine_bin_mins']}
    output_dict['fine_bins_per_week'] = output_dict['fine_bins_per_day'] * 7
    output_dict['fine_bins_per_coarse_bin'] = config_dict['coarse_bin_mins'] // config_dict['fine_bin_mins']
    output_dict['coarse_bins_per_week'] = output_dict['fine_bins_per_week'] // output_dict['fine_bins_per_coarse_bin']
    output_dict['fine_bin_seconds'] = config_dict['fine_bin_mins'] * 60 

    return output_dict

def get_train_test_split(config_dict=config_dict):
    ''' 
    Calculates the fine bin index of the train test and validation period start and end
    Returns a dictionary with all this information in
    '''

    fine_bins_per_day = 24 * 60 // config_dict['fine_bin_mins']

    train_bins = fine_bins_per_day * config_dict['train_days']
    burn_in_bins = fine_bins_per_day * config_dict['burn_in_days']
    validation_bins = fine_bins_per_day * config_dict['validation_days']
    test_bins = fine_bins_per_day * config_dict['test_days']

    # Constructing a dict for the output
    output_dict = {'train_start' : 0,
                   'train_end' : train_bins}
    
    output_dict['burn_in_start'] = output_dict['train_end']
    output_dict['burn_in_end'] = output_dict['burn_in_start'] + burn_in_bins

    output_dict['validation_start'] = output_dict['burn_in_end']
    output_dict['validation_end'] = output_dict['validation_start'] + validation_bins

    output_dict['test_start'] = output_dict['validation_end']
    output_dict['test_end'] = output_dict['test_start'] + test_bins
    
    return output_dict
    
# Adding the fine_bins_per_coarse_bin to the train_test_split_dict
def add_training_denom(bin_metric_dict, config_dict=config_dict):
    ''' 
    Finds how many fine bins are used for the estimate of the iniatial parameter.
    This is the denominator on the mean estimate as it is counts/bins and is similarly used in the variance estimate.

    This fucntion assumes train days is a multiple of 7 to work

    Returns:
        An updated bin metric dict with a new column train denom
    '''
    assert config_dict['train_days'] % 7 == 0
    bin_metric_dict['train_denom'] = bin_metric_dict['fine_bins_per_coarse_bin'] * (config_dict['train_days'] // 7)
    return bin_metric_dict

In [6]:
# Uses the above functions to get 2 dictionaries 
# train_test_dict with the fine bins index upon which we make the train test split
# bin metric dict which calculates metric such as fine bins per day and fine bins used in the training estimate
train_test_dict = get_train_test_split()
bin_metric_dict = get_bin_metrics()
bin_metric_dict = add_training_denom(bin_metric_dict)

### Preprocessing data

This involves:

    - Turning the source_user@domain column into a user_id to make it more lightweight
    - Making the fine bin column and getting counts in each fine bin
    - Making the course bin column
    - Getting the fine bin within coarse bin position

In [7]:
# Creating a fine bin and identifying users with some counts in the bin

def create_counts_data(df):
    ''' 
    Gets the counts per fine bin x source user from the data
    '''
    df = df.with_columns(time = pl.col('time').dt.total_seconds())
    df = df.with_columns(fine_bin_id = pl.col('time') // bin_metric_dict['fine_bin_seconds'])
    user_x_fine_bin_cnts = df.group_by(['source_user@domain', 'fine_bin_id']).agg(pl.len().alias('count')).collect()
    return user_x_fine_bin_cnts

def create_user_to_id_mapping(users_df, mapping_file_name):
        ''' 
        Creates a table containing the 
        '''
        # Creating a user lookup table and storing it
        user_mapping = users_df.select('source_user@domain').unique().sort(by='source_user@domain').with_row_index('user_id')
        user_mapping = user_mapping.with_columns(source_user_type = pl.when(pl.col('source_user@domain').str.contains(r"^U\d+@")).then(pl.lit("human")
                                                ).when(pl.col('source_user@domain').str.contains(r"^C\d+\$@")).then(pl.lit("machine")))
        store_data(user_mapping, mapping_file_name)
        
        # Joining on the lookup table and dropping columns
        users_df = users_df.join(user_mapping, on='source_user@domain', how='inner')
        users_df = users_df.select(['user_id', 'fine_bin_id', 'count']).sort(['user_id', 'fine_bin_id'])

        return users_df, user_mapping

def create_coarse_bins(users_df):
        ''' 
        takes a DF and creates two new columns:
            coarse_bin_id
            fine_bin_within_coarse_pos
        '''
        # Creating columns needed fr
        users_df = users_df.with_columns(fine_bin_pos_in_week = pl.col('fine_bin_id') % bin_metric_dict['fine_bins_per_week'])
        users_df = users_df.with_columns(coarse_bin_id = pl.col('fine_bin_pos_in_week') // bin_metric_dict['fine_bins_per_coarse_bin'])
        return users_df

def create_first_last_interaction_arrays(user_counts):
    '''
    Creates an output table with user first and last interaction indicies within df counts
    Note this is not a fine bin index but a row index
    '''
    # Creating a lookup table for numba for the first and last entry of users 
    user_interactions = user_counts.with_row_index().group_by('user_id').agg(
        user_first_index = pl.min('index'),
        user_last_index = pl.max('index')).sort(by='user_id')
    
    return user_interactions

In [8]:
# Doing initial manipulations on the data including replacing source_user@domain with user_id and constructing course and fine bins
user_counts = create_counts_data(df)
user_counts, user_mapping = create_user_to_id_mapping(user_counts, mapping_file_name='source_users_to_id_mapping')
user_counts = create_coarse_bins(user_counts)
user_interactions = create_first_last_interaction_arrays(user_counts=user_counts)

### Getting inital parameter estimates and interpolation weights

In [9]:
def get_training_sums(user_counts, train_test_dict=train_test_dict):
    ''' 
    Creates the training data and the sum of counts and sum of counts squared columns

    This data is used to get the initial parameter estimates
    '''

    train_df = user_counts.filter((pl.col('fine_bin_id') >= train_test_dict['train_start'])
                                    & (pl.col('fine_bin_id') < train_test_dict['train_end']))

    train_df = train_df.with_columns(count_2 = pl.col('count') ** 2)
    train_df = train_df.group_by(['user_id', 'coarse_bin_id']).agg(pl.sum('count').alias('sum_cnt'), pl.sum('count_2').alias('sum_cnt_2'))
    
    return train_df 


def create_init_grids(user_counts, n_users, n_coarse_bins, bin_metric_dict=bin_metric_dict):
    ''' 
    Creates the u and v init grids from the training data
    '''
    # Getting the sum of counts and sum of counts squared needed for mean and variance calculations
    train_df = get_training_sums(user_counts)
    
    # Init a grid of parmeters to use
    u_init = np.zeros((n_users, n_coarse_bins))
    v_init = np.zeros((n_users, n_coarse_bins))

    # Extrating the entries to assign and assigning them to the df
    entries_to_assign = train_df.select(['user_id', 'coarse_bin_id']).to_numpy()

    u_init[entries_to_assign[:,0], entries_to_assign[:,1]] = train_df['sum_cnt'].to_numpy() / bin_metric_dict['train_denom']
    v_init[entries_to_assign[:,0], entries_to_assign[:,1]] = (train_df['sum_cnt_2'].to_numpy() - 
                                                            ((train_df['sum_cnt'] **2) / bin_metric_dict['train_denom']))/(bin_metric_dict['train_denom'] - 1)

    # Capping the min values of u_init and v_init
    u_init = np.maximum(u_init, config_dict['mean_min'])
    v_init = np.maximum(v_init, config_dict['var_min'])

    return u_init, v_init

In [10]:
# Creatung the inital grids and saving the outputs
u_init, v_init = create_init_grids(user_counts, n_users = user_mapping.shape[0], n_coarse_bins = bin_metric_dict['coarse_bins_per_week'])

### Getting interpolation weights

In [11]:
def get_interpolation_weights(bin_metric_dict=bin_metric_dict):
    '''
    A function that returns the weights we apply when interpolating.
    To understand the computation steps see pages 11 and 12 of lambert liu
    returns:
        weights a numpy array which will be applied as w-1 U-1 + w0 U0 + w1 U1
        the weights array has a row for every fine bin in the coarse bin and 3 columns where each column is the weight being applies to Uis
    '''

    # Getting the m (fine bin number, q and r (defined in lambert liu))
    M = bin_metric_dict['fine_bins_per_coarse_bin']
    m = np.arange(1, M+1)
    q = (m - 1)/M
    r = m/M

    # Getting the two terms used in all calculations
    term_1 = r**2 + r*q + q**2
    term_2 = r + q

    # Computing the weights using the lambert and liu formula
    # these come from rearranging the fomula at the top of page 12 for U-1 U0 and U1
    weights = np.zeros((M, 3))

    weights[:, 0] = term_1/6 - term_2/2 + 1/3
    weights[:, 1] = -term_1/3 + term_2/2 + 5/6
    weights[:, 2] = term_1/ 6 - 1/6

    return weights

In [12]:
# Saving the outputs of this section
interpolation_weights = get_interpolation_weights()

### Getting raw and clustered model

In [13]:
def make_raw_model():
    '''
    Creates a dictionary with the model config to be used by the numba runner
    '''
    n_users, n_coarse_bins = u_init.shape
    # Creating a model dict with dummy placeholders for the cluster mean as no smoothing is done for this model
    # There is 1 cluster so cluster mean vectors have first dim 1
    output = {'name' : 'raw_model',
              'cluster_param' : 0,
              'cluster_mean_u' : np.zeros((1,n_coarse_bins), dtype='float64'),
              'cluster_mean_v' : np.zeros((1,n_coarse_bins), dtype='float64'),
              'smoothing_strength' : 0,
              'cluster_assignments' : np.zeros(n_users, dtype='int64')
              }
    
    return output

raw_model = make_raw_model()

#### Clustering methods

- Cell 1 matrix to cluster construction
- Cell 2 Clustering method choice function
- Cell 3 Cluster summary statistics helpers

In [14]:
#### Creating functions that return the correct vector for clustering

# Matrix construction functions
def make_u_matrix(u, v): return u

def make_log_u_matrix(u, v): return np.log(u)

def make_v_matrix(u, v): return v

def make_normalised_u_clustering_matrix(u, v): return u / u.sum(axis=1)[:, None]


# Wrapper function
def make_clustering_matrix(u, v, config_dict=config_dict): 
    ''' 
    Constructs the matrix used for clustering. 
    The construction used depends on `clustering_matrix_name` found within `config_dict`
    '''
    if config_dict['clustering_matrix_name'] == 'u':
        return make_u_matrix(u, v)
    elif config_dict['clustering_matrix_name'] == 'log_u':
        return make_log_u_matrix(u, v)
    elif config_dict['clustering_matrix_name'] == 'v':
        return make_v_matrix(u, v)
    elif config_dict['clustering_matrix_name'] == 'normalised_u':
        return make_normalised_u_clustering_matrix(u, v)
    else: 
        raise ValueError("config_dict['clustering_matrix_name'] invalid")
    



In [15]:
### Creating clusters and then initialising a cluster model

def get_k_means_assignments(k, random_state, matrix_to_cluster):
    '''
    Performs k means clustering on a numpy array and returns a vector of cluster assignments
    '''

    k_means_model = KMeans(n_clusters=k, random_state=random_state)
    clusters = k_means_model.fit_predict(matrix_to_cluster).astype(np.int64)

    return clusters, k_means_model

def get_cluster_assignments(cluster_param, matrix_to_cluster, config_dict=config_dict):
    ''' 
    Generic clustering runner that can be scaled to include multiple algorithms
    '''
    if config_dict['clustering_method'] == 'k_means':
        return get_k_means_assignments(k=cluster_param, random_state=config_dict['seed'], matrix_to_cluster=matrix_to_cluster)
    else:
        raise ValueError('Clustering method not Reckognised')

In [16]:
def get_centroid_distance(cluster_centres):
    ''' 
    For each cluster computes l2 distances and returns:
        - Average distance to other clusters
        - Min distance to other clusters
    '''
    output = {}
    for i in range(cluster_centres.shape[0]):
        cluster_distances = []
        for j in range(cluster_centres.shape[0]):
            
            # Calculate l2 distance
            if i != j:
                cluster_distances.append(np.sqrt(np.sum((cluster_centres[i] - cluster_centres[j]) ** 2)))

        # Get outputs for cluster
        output[i] = {'nearest_centroid_dist' : min(cluster_distances), 'avg_centroid_dist' : sum(cluster_distances)/len(cluster_distances)}

    return output

def create_cluster_summary_df(model, user_mapping=user_mapping, config_dict=config_dict):
    ''' 
    Creates an output df which summarises cluster quality metrics
    '''

    # Getting the number of clusters
    if config_dict['clustering_method'] == 'k_means':
        n_clusters=model['cluster_param']
    else:
        raise ValueError('Clustering method not Reckognised')
    
    output = []
    cluster_centroid_dict = get_centroid_distance(model['cluster_centres'])

    for cluster_id in range(n_clusters):

        users_in_cluster = user_mapping.filter(pl.col('user_id').is_in(np.where(model['cluster_assignments'] == cluster_id)[0]))
        n_users_in_cluster = users_in_cluster.shape[0]
        human_users_in_cluster = users_in_cluster.filter(pl.col('source_user_type') == 'human').shape[0]
        machine_users_in_cluster = users_in_cluster.filter(pl.col('source_user_type') == 'machine').shape[0]

        output.append({
            'cluster_id' : cluster_id,
            'cluster_param' : model['cluster_param'],
            'seed' : model['seed'],
            'clustering_matrix_name' : model['clustering_matrix_name'],

            # Cluster content metrics
            'n_users' : n_users_in_cluster,
            'n_humans' : human_users_in_cluster,
            'n_machine_users' : machine_users_in_cluster,

            # Performance metrics
            'inertia' : model['cluster_inertia'],
            'nearest_centroid_dist' : cluster_centroid_dict[cluster_id]['nearest_centroid_dist'],
            'avg_centroid_dist' : cluster_centroid_dict[cluster_id]['avg_centroid_dist'],
        })

    return pl.DataFrame(output)


In [ ]:
## Clustering model and getting cluster means helper 
#! NOTE also we should probably cluster the data based on the parameters for the full training data not just for the first week of train

def get_clustering_means(cluster_groups, u_init=u_init, v_init=v_init):
    ''' 
    Calculates mean u and v values for each cluster group and each time bin
    Used within the get clustering model function
    Args:
        cluster_groups a n_users length vector of cluster assignments
        u_init : the calculated vector of parameter means
        v_init : the calculated vector of initial parameter variances
    '''

    n_users, n_coarse_bins = u_init.shape 
    n_clusters = cluster_groups.max() + 1
    print(f'Number of clusters identified : {n_clusters}')

    # Init mean vectors
    cluster_mean_u = np.zeros((n_clusters, n_coarse_bins), dtype='float64')
    cluster_mean_v = np.zeros((n_clusters, n_coarse_bins), dtype='float64')
    users_per_cluster = np.zeros(n_clusters, dtype='float64')

    # Summing u and v contributions in each cluster
    # Extract the cluster assignment for each user and then add their parameters to each bin
    for user_id in range(n_users):
        cluster_assignment = int(cluster_groups[user_id])
        cluster_mean_u[cluster_assignment, :] += u_init[user_id, :]
        cluster_mean_v[cluster_assignment, :] += v_init[user_id, :]
        users_per_cluster[cluster_assignment] += 1

    # Dividing through to get the averages in each cluster
    for cluster_assignment in range(n_clusters):
        if users_per_cluster[cluster_assignment] > 0:
            cluster_mean_u[cluster_assignment, :] /= users_per_cluster[cluster_assignment]
            cluster_mean_v[cluster_assignment, :] /= users_per_cluster[cluster_assignment]

    return cluster_mean_u, cluster_mean_v

def make_cluster_model(cluster_param, config_dict, u_init=u_init, v_init=v_init, smoothing_strength=config_dict['smoothing_strength']):
    ''' 
    Creates the clustering model dictionary 
    '''
    matrix_to_cluster = make_clustering_matrix(u_init, v_init, config_dict=config_dict)
    cluster_assignments, clustering_model = get_cluster_assignments(cluster_param=cluster_param, matrix_to_cluster=matrix_to_cluster, config_dict=config_dict)
    cluster_mean_u, cluster_mean_v = get_clustering_means(u_init=u_init, v_init=v_init, cluster_groups=cluster_assignments)


    output = {
        # Clustering configs
        'name' : f"{config_dict['clustering_method']}",
        'clustering_matrix_name' : config_dict['clustering_matrix_name'],
        'seed' : config_dict['seed'],
        'cluster_param' : cluster_param,
        'smoothing_strength' : smoothing_strength,

        # Identified values
        'cluster_mean_u' : cluster_mean_u,
        'cluster_mean_v' : cluster_mean_v,
        'cluster_assignments' : cluster_assignments,
        
        # Cluster quality metrics
        'cluster_inertia' : clustering_model.inertia_, 
        'cluster_centres' : clustering_model.cluster_centers_,
        }

    return output 

In [18]:
# Getting the clustering model summarising cluster performance and storing the df
clustering_model = make_cluster_model(cluster_param=4, config_dict=config_dict)
summary_df = create_cluster_summary_df(clustering_model)
store_data(summary_df, 'clustering_algorithm_performance', csv=True)

Number of clusters identified : 4


### Storing data and converting to nt for final runner

- Eventually will split the notebook here into preprocessing and the rest so we just load the data and ll

In [ ]:
def dictionary_to_named_tuple_class(name : str, dictionary : dict):
    ''' 
    Converts a dict to a named tuple with the given name.
    Can be used downstream in the njit functions
    '''
    if name in globals():
        raise ValueError(f'Named tuple: {name} already exists')
    else:
        return namedtuple(name, dictionary.keys())
        
    
def df_to_nt(name, df):
    ''' 
    Converts a dataframe to a Namedtuple of numpy arrays for use in the numba runner
    '''
    table_dict = {col : df[col].to_numpy().astype('int64') for col in df.columns}

    return dictionary_to_named_tuple_class(name, table_dict)(**table_dict)

In [20]:
# Storing data needed for the numba runner
user_counts = user_counts.select(['user_id', 'fine_bin_id', 'count'])
store_data(user_interactions, 'user_interactions')
store_data(user_counts, 'user_counts')
store_data(interpolation_weights, filename='interpolation_weights')
store_data(u_init, 'u_init')
store_data(v_init, 'v_init')

##### Creating NT LL args

In [21]:
# Converting the dfs to nt of numpy arrays to be used for the final numba runner
user_interactions_nt = df_to_nt('user_interactions_nt', user_interactions)
user_counts_nt = df_to_nt('user_counts_nt', user_counts)

In [ ]:
# Creating dictionaries of names of outputs and oder they appear in
output_names = ['n_bins_scored', 'n_degen_bins', 'non_degen_ll_sum','non_degen_smoothed_ll_sum','degen_bins_w_counts_observed', 
                'degen_bin_raw_pred_activity', 'degen_bin_smoothed_pred_activity']
model_names = ['raw_model_calib_index', 'smoothed_model_calib_index']

output_idx_dict = {name : idx for idx, name in enumerate(output_names)}
model_idx_dict = {name : idx for idx, name in enumerate(model_names)}

# Converting userful info to named tuples
## Creating a seperate config_nt_class and train_test_nt_class as they are reused in the tuning loop
config_nt_class = dictionary_to_named_tuple_class('config_nt', config_dict)
config_nt = config_nt_class(**config_dict)
train_test_nt_class = dictionary_to_named_tuple_class('train_test_nt', train_test_dict)
train_test_nt = train_test_nt_class(**train_test_dict)

# For other dicts we dont need to save the class
output_idx_nt = dictionary_to_named_tuple_class('output_idx_nt', output_idx_dict)(**output_idx_dict)
model_idx_nt = dictionary_to_named_tuple_class('model_idx_nt', model_idx_dict)(**model_idx_dict)
bin_metric_nt = dictionary_to_named_tuple_class('bin_metric_nt', bin_metric_dict)(**bin_metric_dict)

### Creating helper functions for the final runner

#### Math helper functions

In [23]:
# Creating functions that get the log pmf value for the negative binomial distribution (or the poisson distirbution for underdispersed users)
from numba import njit
import math 

### Helper functions to stop overflow
@njit(inline='always')
def logsumexp2(a, b):
    ''' 
    Two term logsumexp helper
    '''
    if a == b:
        return a + math.log(2.0)
    if a > b:
        return a + math.log1p(math.exp(b - a))
    else:
        return b + math.log1p(math.exp(a - b))


log_0_5 = -math.log(2.0)

@njit(inline="always")
def log1minexp(log_p):
    """
    Stable log(1 - exp(log_p))
    """

    if log_p < log_0_5:
        return math.log1p(-math.exp(log_p))
    else:
        return math.log(-math.expm1(log_p))

### LOG PMF VALUES
@njit 
def poisson_lpmf(x, mu):
    ''' 
    Poisson log pmf
    '''
    return x*math.log(mu) - mu - math.lgamma(x+1)

@njit
def neg_bin_lpmf(x, mu, sigma2):
    p = mu/sigma2
    r = (mu*p) / (1-p)
    return math.lgamma(x+r) - math.lgamma(r) - math.lgamma(x+1) + r*math.log(p) + x*math.log(1-p)

@njit 
def get_lpmf_val(x, mu, sigma2, mean_min, var_min, min_mean_var_diff):
    if mu <= 0:
        raise ValueError(' Mu < 0 ')
    if sigma2 <= 0:
        raise ValueError('Sigma^2 , 0')
    
    mu = max(mu, mean_min)
    sigma2 = max(sigma2, var_min)
    if sigma2 <= mu + min_mean_var_diff:
        return poisson_lpmf(x, mu)
    else: 
        return neg_bin_lpmf(x, mu, sigma2)

### UPPER TAIL VALUES
@njit 
def poisson_log_upper_tail(x, mu):
    ''' 
    Gets p(X>= x) for a poisson distribution 
    '''
    if x == 0:
        return 0
    
    # Looping over k and getting prob x = k and adding to lower tail
    log_prob_k = -mu
    lower_tail = log_prob_k
    for k in range(1,x):
        log_prob_k = log_prob_k + math.log(mu) - math.log(k)
        lower_tail = logsumexp2(lower_tail, log_prob_k)

    return log1minexp(lower_tail)


@njit 
def neg_bin_log_upper_tail(x, mu, sigma2):
    ''' 
    Gets p(X>= x) for a negative binomial distribution 
    '''
    if x == 0:
        return 0
    
    p = mu/sigma2
    r = (mu*p) / (1-p)

    # Looping over k and getting prob x = k and adding to lower tail
    log_prob_k = r * math.log(p)
    log_lower_tail = log_prob_k
    for k in range(1,x):
        log_prob_k = log_prob_k+ math.log((k-1)+r) - math.log(k) + math.log(1-p)
        log_lower_tail =  logsumexp2(log_lower_tail, log_prob_k)

    return log1minexp(log_lower_tail)

@njit 
def get_upper_tail_value(x, mu, sigma2, mean_min, var_min, min_mean_var_diff):
    ''' 
    We dont need edge case checks here as the other function is called with the same mu and sigma
    '''
    mu = max(mu, mean_min)
    sigma2 = max(sigma2, var_min)
    if sigma2 <= mu + min_mean_var_diff:
        return poisson_log_upper_tail(x, mu)
    else: 
        return neg_bin_log_upper_tail(x, mu, sigma2)

#### Helper function for metrics

In [24]:
@njit
def update_ouputs(x, time_period_int, output_metrics, calibration_output, output_idx_nt, model_idx_nt,
                  log_calibration_thresholds, log_degen_threshold,
                  log_p0_raw, log_p0_smoothed, log_upper_tail_raw, log_upper_tail_smoothed, lpmf_raw, lpmf_smoothed):
    ''' 
    Function used for updating the output and calibration threshold output
    Args:
        x : the count observed
        time_period_int : number that states whether we are in test train or validation
        output_metrics : Np array where we will store our outputs
        calibration_output : output of how many p values fall below each threshold in train and validation for each model
        output_idx_nt : named tuple that tells us which index of `output_metrics` each metric lives in
        model_idx_nt :  named tuple that tells us which index of `calibration_output` each model lives in

        log_raw_degen_threshold : threshold that tells us whether we are in a degenerate bin or not (defined by P(X=0))
        log_calibration_thresholds : calibration thresholds we are monitoring (how many p values fall below each threshold)

        log_p0_raw  + smoothed: the log prob of 0 is compared to the degen threshold in the function
        log_upper_tail_raw + smooth : the log_probability of observing a value greater that or equal to x. compared to the log calibration thresholds

        lpmf_raw + smoothed : the log likelihood values we observe
    '''

    # If we are in train or validation update the metrics
    if time_period_int == 0 or time_period_int == 1:
        output_metrics[time_period_int, output_idx_nt.n_bins_scored] += 1

        # Check if we are in a degen bin and then update the degen bin
        if log_p0_raw > log_degen_threshold:
            output_metrics[time_period_int, output_idx_nt.n_degen_bins] += 1

            if x > 0:
                output_metrics[time_period_int, output_idx_nt.degen_bins_w_counts_observed] += 1

            output_metrics[time_period_int, output_idx_nt.degen_bin_raw_pred_activity] += -math.expm1(log_p0_raw)
            output_metrics[time_period_int, output_idx_nt.degen_bin_smoothed_pred_activity] += -math.expm1(log_p0_smoothed)

        # If we are not in a degenerate bin update log likelihood and 
        else:
            # Update the log likelihood function
            output_metrics[time_period_int, output_idx_nt.non_degen_ll_sum] += lpmf_raw
            output_metrics[time_period_int, output_idx_nt.non_degen_smoothed_ll_sum] += lpmf_smoothed

            # Add a point for each calibration threshold we are less than 
            # 1 row of output for raw model one for smoothed model
            for calib_threshold_idx in range(log_calibration_thresholds.shape[0]):

                if log_upper_tail_raw < log_calibration_thresholds[calib_threshold_idx]:
                    calibration_output[time_period_int, calib_threshold_idx, model_idx_nt.raw_model_calib_index] += 1

                if log_upper_tail_smoothed < log_calibration_thresholds[calib_threshold_idx]:
                    calibration_output[time_period_int, calib_threshold_idx, model_idx_nt.smoothed_model_calib_index] += 1

#### Time period function

In [25]:
@njit
def get_time_period(fine_bin_id, validation_start, validation_end, test_start, test_end):
    ''' 
    Returns:
        0 if we are in the validation data
        1 if we are in the test data
        -1 otherwise (train + burn in and any unused data)
    '''
    if validation_start <= fine_bin_id and fine_bin_id < validation_end:
        return 0
    elif test_start <= fine_bin_id and fine_bin_id < test_end:
        return 1
    else: 
        return -1

#### Smooting helpers

In [26]:
# Creating a function that smooths between users and parameter
@njit 
def smoothing_function(smoothing_strength, user_parameter, cluster_parameter):
    ''' 
    Function used for smoothing between the cluster parameter and the user parameter
    '''
    return (1-smoothing_strength) * user_parameter + smoothing_strength*cluster_parameter

@njit 
def interpolate_values(v_neg_1, v_0, v_1, fine_bin_within_coarse_pos, interpolation_weights):
    '''
        Applies the quadratic interpolation between the left middle and right bin values
    '''
    # Extract the weights we will use for the interpolation
    w_neg_1, w_0, w_1 = interpolation_weights[fine_bin_within_coarse_pos]
    return w_neg_1 * v_neg_1 + w_0 * v_0 + w_1 * v_1

@njit 
def smooth_params(user_param_grid, cluster_param_grid, cluster_assignments, smoothing_strength, crnt_user, crnt_coarse_bin,
                  crnt_fine_bin_within_coarse_pos, interpolation_weights):
    ''' 
    Takes a parameter mu or sigma2 and smooths it towards the cluster parameter using `smoothing_function`
    '''
    # Get the cluster assignment for the current user
    cluster_id = cluster_assignments[crnt_user]

    # Get the 3 values to interpolate
    n_coarse_bins = user_param_grid.shape[1]
    neg_1_coarse_bin = (crnt_coarse_bin -1) % n_coarse_bins
    _1_coarse_bin = (crnt_coarse_bin + 1)% n_coarse_bins

    ## Smooth the 3 values we will later interpolate
    v_neg_1 = smoothing_function(smoothing_strength, user_param_grid[crnt_user, neg_1_coarse_bin], cluster_param_grid[cluster_id, neg_1_coarse_bin])
    v_0 = smoothing_function(smoothing_strength, user_param_grid[crnt_user, crnt_coarse_bin], cluster_param_grid[cluster_id, crnt_coarse_bin])
    v_1 = smoothing_function(smoothing_strength, user_param_grid[crnt_user, _1_coarse_bin], cluster_param_grid[cluster_id, _1_coarse_bin])

    # Interpolate the values 
    return interpolate_values(v_neg_1, v_0, v_1, crnt_fine_bin_within_coarse_pos, interpolation_weights)



@njit
def get_smoothed_params(u, v, cluster_u, cluster_v, cluster_assignments, smoothing_strength, crnt_user, crnt_coarse_bin,
                        crnt_fine_bin_within_coarse_pos, interpolation_weights):
    
    mu = smooth_params(u, cluster_u, 
                       cluster_assignments=cluster_assignments, smoothing_strength=smoothing_strength, crnt_user=crnt_user, 
                       crnt_coarse_bin=crnt_coarse_bin, crnt_fine_bin_within_coarse_pos=crnt_fine_bin_within_coarse_pos, interpolation_weights=interpolation_weights)
    
    sigma2 = smooth_params(v, cluster_v, 
                        cluster_assignments=cluster_assignments, smoothing_strength=smoothing_strength, crnt_user=crnt_user, 
                        crnt_coarse_bin=crnt_coarse_bin, crnt_fine_bin_within_coarse_pos=crnt_fine_bin_within_coarse_pos, interpolation_weights=interpolation_weights)

    return mu, sigma2

#### Creating new grid and new cluster means

In [27]:
# Creating a function that collects grid updates and a function that updates the grid

@njit
def update_grid(u, v, crnt_user_id, usr_updt_u_sum, usr_updt_v_sum, fine_bins_per_coarse_bin):
    ''' 
    Replaces the grid values using the temporary grid as data comes in
    '''
    u[crnt_user_id, : ] = usr_updt_u_sum/fine_bins_per_coarse_bin
    v[crnt_user_id, : ] = usr_updt_v_sum/fine_bins_per_coarse_bin

@njit
def collect_temp_grid(usr_updt_u_sum, usr_updt_v_sum, crnt_coarse_bin, x, mu_t, sigma_2_t, w, mean_min, var_min):
    '''
    As data comes in we update the interpolated mu values by combining with incoming data as per lambert and liu formula
    returns nothing as we modify in place
    '''
    mu_new = (1-w)*mu_t + w*x
    sigma2_new = (1-w)*sigma_2_t + w*(x-mu_t)*(x-mu_new)

    mu_new = max(mu_new, mean_min)
    sigma2_new = max(sigma2_new, var_min)

    usr_updt_u_sum[crnt_coarse_bin] += mu_new
    usr_updt_v_sum[crnt_coarse_bin] += sigma2_new
    

In [28]:
# Function for updating cluster parameters 
@njit 
def get_new_clustering_means(cluster_groups, u, v):
    ''' 
    Calculates mean u and v values for each cluster group and each time bin
    Args:
        cluster_groups a n_users length vector of cluster assignments
        u : the calculated vector of u parameter means
        v : the calculated vector of v parameter variances
    '''

    n_users, n_coarse_bins = u.shape 
    n_clusters = cluster_groups.max() + 1

    # Init mean vectors
    cluster_mean_u = np.zeros((n_clusters, n_coarse_bins), dtype=np.float64)
    cluster_mean_v = np.zeros((n_clusters, n_coarse_bins), dtype=np.float64)
    users_per_cluster = np.zeros(n_clusters, dtype=np.float64)

    # Summing u and v contributions in each cluster
    # Extract the cluster assignment for each user and then add their parameters to each bin
    for user_id in range(n_users):
        cluster_assignment = int(cluster_groups[user_id])
        cluster_mean_u[cluster_assignment, :] += u[user_id, :]
        cluster_mean_v[cluster_assignment, :] += v[user_id, :]
        users_per_cluster[cluster_assignment] += 1

    # Dividing through to get the averages in each cluster
    for cluster_assignment in range(n_clusters):
        if users_per_cluster[cluster_assignment] > 0:
            cluster_mean_u[cluster_assignment, :] /= users_per_cluster[cluster_assignment]
            cluster_mean_v[cluster_assignment, :] /= users_per_cluster[cluster_assignment]

    return cluster_mean_u, cluster_mean_v

#### Lambert liu runner

In [ ]:
@njit
def run_lambert_liu(u_init, v_init, cluster_u_init, cluster_v_init, cluster_groups, smoothing_strength, 
                    user_counts_nt, user_interactions_nt, interpolation_weights,
                    train_test_nt, bin_metric_nt, config_nt,
                    output_idx_nt, model_idx_nt):
    ''' 
    Runs the lambert liu algorithm
    Args:
        u_init + v_init : inital parameter grids
        cluster_u_init + cluster_v_init : inital cluster parameters
        cluster groups : inital cluster assignments 1 row per user id
        smoothing_strength : parameter (experiment to vary)
        user_counts_nt: is a named tuple version of user_counts has columns user_id, fine_bin_id, count
        user_interactions_nt: is a named tuple verion of user_interactions has columns user id and first and last interaction index in user_counts
        interpolation_weights : precalculated weights for parameter interpolation
        train_test_nt, bin_metric_nt, config_dt : named tuple versions of dicts train_test_dict, config_dict and bin_metric_dict
        output_idx_nt : a named tuple containing the names and indicies of the outputs we want to store
        model_idx_nt : a named tuple containing the names and indicies where we store each model outputs
    '''
    ###
    # Initialising grids where we will keep track of parameters and cluster mean parameters
    u = u_init.copy()
    v = v_init.copy()

    cluster_u = cluster_u_init.copy()
    cluster_v = cluster_v_init.copy()

    # Extracting bin and user numbers
    n_users, n_coarse_bins = u.shape

    # Create log_calibration_thresholds and degen threshold
    log_calibration_thresholds = np.log(config_nt.calibration_thresholds)
    log_degen_threshold = np.log(config_nt.degen_threshold)

    # Creating a dict for output 1 row for validation and 1 row for test
    # Columns are number of entries, log likelihood, the observations and the interpolated parameters
    output_metrics = np.zeros((2, len(output_idx_nt)), dtype='float64')
    calibration_output = np.zeros((2, log_calibration_thresholds.shape[0], len(model_idx_nt)), dtype='float64')

    # Getting the weeks to iterate over in the data
    # -1 to avoid an empty last week
    burn_in_first_week = train_test_nt.burn_in_start // bin_metric_nt.fine_bins_per_week
    test_last_week = (train_test_nt.test_end-1)// bin_metric_nt.fine_bins_per_week
    ###

    ##
    # Initialising a numpy array with 1 row per user which contains an index
    # the index points to the users first burn in row of counts df
    usr_frst_rw = user_interactions_nt.user_first_index.copy()

    for user_id in range(n_users):
        cnt_tbl_idx = user_interactions_nt.user_first_index[user_id]
        usr_lst_idx = user_interactions_nt.user_last_index[user_id]
        
        while cnt_tbl_idx <= usr_lst_idx and user_counts_nt.fine_bin_id[cnt_tbl_idx] < train_test_nt.burn_in_start:
            cnt_tbl_idx +=1
        usr_frst_rw[user_id] = cnt_tbl_idx
    ##

    for week in range(burn_in_first_week, test_last_week + 1):
        
        week_start = week * bin_metric_nt.fine_bins_per_week
        week_end = (week + 1) * bin_metric_nt.fine_bins_per_week

        if week_end > train_test_nt.test_end:
            week_end = train_test_nt.test_end

        # For each week iterate over the users and init the pointers
        for user_id in range(n_users):

            cnt_tbl_idx = usr_frst_rw[user_id]
            usr_end_idx = user_interactions_nt.user_last_index[user_id]

            # Init numpy vectors for calculating the user sums
            usr_updt_u_sum = np.zeros(n_coarse_bins, dtype=np.float64)
            usr_updt_v_sum = np.zeros(n_coarse_bins, dtype=np.float64)

            for fine_bin_idx in range(week_start, week_end):
                # If we have a count in this bin make it x else make it 0
                # Move our current able index pointer
                if cnt_tbl_idx <= usr_end_idx and user_counts_nt.fine_bin_id[cnt_tbl_idx] == fine_bin_idx:
                    x = user_counts_nt.count[cnt_tbl_idx]
                    cnt_tbl_idx += 1
                else:
                    x = 0

                # Getting bin metrixs
                fine_bin_pos_in_week = fine_bin_idx % bin_metric_nt.fine_bins_per_week
                crnt_coarse_bin = fine_bin_pos_in_week // bin_metric_nt.fine_bins_per_coarse_bin
                crnt_fine_bin_within_coarse_pos = fine_bin_pos_in_week % bin_metric_nt.fine_bins_per_coarse_bin

                # Getting the smoothed params and capping them at the minimal value
                mu_t, sigma_2_t = get_smoothed_params(u, v, cluster_u, cluster_v, cluster_groups, smoothing_strength, user_id, crnt_coarse_bin, 
                                                     crnt_fine_bin_within_coarse_pos, interpolation_weights)
                mu_t = max(mu_t, config_nt.mean_min)
                sigma_2_t = max(sigma_2_t, config_nt.var_min)

                # Updating validation and test metrics

                # Getting unsmoothed but interpolated params and using that for updates (difference from above call is passing smoothing strength 0):
                mu_unsmth_t, sigma_unsmth_2_t = get_smoothed_params(u, v, cluster_u, cluster_v, cluster_groups, 0, user_id, crnt_coarse_bin, 
                                                     crnt_fine_bin_within_coarse_pos, interpolation_weights)
                mu_unsmth_t = max(mu_unsmth_t, config_nt.mean_min)
                sigma_unsmth_2_t = max(sigma_unsmth_2_t, config_nt.var_min)
                time_period_int = get_time_period(fine_bin_idx, train_test_nt.validation_start, train_test_nt.validation_end, 
                                                                train_test_nt.test_start, train_test_nt.test_end)

                # Getting the LPMF of the observed counts and 0 for both the raw and smoothed value
                # The raw model can be used to determine whether the bin is degenerate
                lpmf_smoothed = get_lpmf_val(x, mu_t, sigma_2_t, config_nt.mean_min, config_nt.var_min, config_nt.min_mean_var_diff)
                lpmf_raw = get_lpmf_val(x, mu_unsmth_t, sigma_unsmth_2_t, config_nt.mean_min, config_nt.var_min, config_nt.min_mean_var_diff)

                if x == 0:
                    log_p0_raw = lpmf_raw
                    log_p0_smoothed = lpmf_smoothed
                else: 
                    log_p0_raw = get_lpmf_val(0, mu_unsmth_t, sigma_unsmth_2_t, config_nt.mean_min, config_nt.var_min, config_nt.min_mean_var_diff)
                    log_p0_smoothed = get_lpmf_val(0, mu_t, sigma_2_t, config_nt.mean_min, config_nt.var_min, config_nt.min_mean_var_diff)

                # Getting the upper tail value for both the raw and the smoothed model
                log_upper_tail_raw = get_upper_tail_value(x, mu_unsmth_t, sigma_unsmth_2_t, config_nt.mean_min, config_nt.var_min, config_nt.min_mean_var_diff)
                log_upper_tail_smoothed = get_upper_tail_value(x, mu_t, sigma_2_t, config_nt.mean_min, config_nt.var_min, config_nt.min_mean_var_diff)

                # Updating outputs
                update_ouputs(x, time_period_int, output_metrics, calibration_output, output_idx_nt, model_idx_nt, log_calibration_thresholds, log_degen_threshold,
                                log_p0_raw, log_p0_smoothed, log_upper_tail_raw, log_upper_tail_smoothed, lpmf_raw, lpmf_smoothed)

                collect_temp_grid(usr_updt_u_sum, usr_updt_v_sum, crnt_coarse_bin, x, mu_unsmth_t, sigma_unsmth_2_t, config_nt.w, config_nt.mean_min, config_nt.var_min)
            
            # Updating the users first row (for the next week)
            usr_frst_rw[user_id] = cnt_tbl_idx

            update_grid(u, v, user_id, usr_updt_u_sum, usr_updt_v_sum, bin_metric_nt.fine_bins_per_coarse_bin)
        cluster_u, cluster_v = get_new_clustering_means(cluster_groups, u, v)

    return output_metrics, calibration_output, u, v, cluster_u, cluster_v

In [ ]:
# running the lambert liu runner
def run_pipeline_ll(model, config_nt, train_test_nt):
    ''' 
    Makes a call to the numba lambert liu runner
    '''

    return run_lambert_liu(
    u_init=u_init,
    v_init=v_init,
    cluster_u_init=model['cluster_mean_u'],
    cluster_v_init=model['cluster_mean_v'],
    cluster_groups=model['cluster_assignments'],
    smoothing_strength=model['smoothing_strength'],
    user_counts_nt=user_counts_nt,
    user_interactions_nt=user_interactions_nt,
    interpolation_weights=interpolation_weights,
    train_test_nt=train_test_nt,
    bin_metric_nt=bin_metric_nt,
    config_nt=config_nt,
    output_idx_nt=output_idx_nt,
    model_idx_nt=model_idx_nt)


In [ ]:
output_metrics, calibration_outputs, u_final, v_final, cluster_u_final, cluster_v_final = run_pipeline_ll(clustering_model, config_nt=config_nt, train_test_nt=train_test_nt)

### Output Table Creation

In [ ]:
def make_output_table_row(model, output_metrics, config_dict, test_valid):
    ''' 
    Transforms the output row into a readable dictionary that can be used as an output df row.
    Args:
        test_valid : Can take values `test` and `valid` tells us what period we are in

    '''

    # Init variables
    if test_valid == 'valid':
        period_idx = 0
    elif test_valid == 'test':
        period_idx = 1
    else:
        raise ValueError('test_valid must either be `test` or `valid`')
    n_non_degen_bins = output_metrics[period_idx, output_idx_nt.n_bins_scored] - output_metrics[period_idx, output_idx_nt.n_degen_bins]
    

    output = {
        # Row descriptions
        'smoothed_model_name': model['name'],
        'w': config_dict['w'],
        'cluster_param': model['cluster_param'],
        'smoothing_strength': model['smoothing_strength'],
        'test_valid' : test_valid,

        # Log likelihood for non degenerate bins
        'non_degen_ll': output_metrics[period_idx, output_idx_nt.non_degen_ll_sum] / n_non_degen_bins,
        'non_degen_smoothed_ll': output_metrics[period_idx, output_idx_nt.non_degen_smoothed_ll_sum] / n_non_degen_bins,

        # Calibration for the degenerate bins
        'degen_activity_rate': output_metrics[period_idx, output_idx_nt.degen_bins_w_counts_observed] / output_metrics[period_idx, output_idx_nt.n_degen_bins],
        'degen_smoothed_pred_activity_rate': output_metrics[period_idx, output_idx_nt.degen_bin_smoothed_pred_activity] / output_metrics[period_idx, output_idx_nt.n_degen_bins],
        'degen_raw_pred_activity_rate': output_metrics[period_idx, output_idx_nt.degen_bin_raw_pred_activity] / output_metrics[period_idx, output_idx_nt.n_degen_bins],
    
        # Bin Metrics
        'n_bins_scored': output_metrics[period_idx, output_idx_nt.n_bins_scored],
        'n_degen_bins': output_metrics[period_idx, output_idx_nt.n_degen_bins],
        'n_non_degen_bins': n_non_degen_bins,
        'degen_share': output_metrics[period_idx, output_idx_nt.n_degen_bins] / output_metrics[period_idx, output_idx_nt.n_bins_scored],
        
        # Clustering metrics
        'clustering_matrix_name': model['clustering_matrix_name'],
        'seed': model['seed'],
        'cluster_inertia': model['cluster_inertia'],}

    return output



def make_calibration_output_rows(model, output_metrics, calibration_outputs, test_valid, config_dict):
    ''' 
    Creates a table of calibration outputs
    '''

    # Init variables and outputs
    if test_valid == 'valid':
        period_idx = 0
    elif test_valid == 'test':
        period_idx = 1
    else:
        raise ValueError('test_valid must either be `test` or `valid`')

    n_non_degen_bins = output_metrics[period_idx, output_idx_nt.n_bins_scored] - output_metrics[period_idx, output_idx_nt.n_degen_bins]
    output = []

    for threshold_idx in range(config_dict['calibration_thresholds'].shape[0]):

        ## Appending a row to output
        output.append({
            # Row descriptions
            'smoothed_model_name': model['name'],
            'w': config_dict['w'],
            'cluster_param': model['cluster_param'],
            'smoothing_strength': model['smoothing_strength'],
            'test_valid': test_valid,

            # Calibration metrics
            'threshold': config_dict['calibration_thresholds'][threshold_idx],
            'observed_raw_tail_rate': calibration_outputs[period_idx, threshold_idx, model_idx_nt.raw_model_calib_index] / n_non_degen_bins,
            'observed_smoothed_tail_rate': calibration_outputs[period_idx, threshold_idx, model_idx_nt.smoothed_model_calib_index] / n_non_degen_bins,
        })

    return output

### Creating a validation hyperparameter tuning loop

In [ ]:
def create_tuning_dict(dict):
    ''' 
    Returns a dictionary without a test period used for hyperparameter tuning
    '''
    validation_only_dict = dict.copy()
    validation_only_dict['test_start'] = validation_only_dict['validation_end']
    validation_only_dict['test_end'] = validation_only_dict['validation_end']
    return validation_only_dict

def tune_models(cluster_param_values, w_values, smoothing_strength, train_test_dict, config_dict, config_nt_class=config_nt_class):
    ''' 
    Runs the hyperparameter tuning for the cluster based smoothing model and the raw model
    '''
    validation_only_dict = create_tuning_dict(train_test_dict)
    validation_only_nt = train_test_nt_class(**validation_only_dict)
    results = []
    for cluster_param in cluster_param_values:
        model = make_cluster_model(cluster_param=cluster_param, smoothing_strength=smoothing_strength, config_dict=config_dict)
        for w in w_values:
            temp_config = config_dict.copy()
            temp_config['w'] = w
            temp_config_nt = config_nt_class(**temp_config)

            output_metrics, *_ = run_pipeline_ll(model, config_nt=temp_config_nt, train_test_nt=validation_only_nt)

            row = make_output_table_row(model, output_metrics, temp_config, test_valid='valid')
            results.append(row)

    return results

In [ ]:
results = tune_models(cluster_param_values=[4,8], w_values=[0.1, 0.01], smoothing_strength=0.2,
                                train_test_dict=train_test_dict, config_dict=config_dict)

Number of clusters identified : 4
Number of clusters identified : 8


## Creating a simple benchmark for comparison

The benchmark we will use is an empirical CDF which looks at the counts 1 hour either side of the data point in the previous n weeks and then uses that to make an ECDF for the model

In [28]:
@njit
def get_simple_benchmark_performance(cnts_tbl_f_bn_id, cnts_tbl_cnt, intract_tbl_frst_int, intract_tbl_lst_int, validation_start, validation_end, 
                                     test_start, test_end, fine_bins_per_week, window_radius_fb, historical_weeks_used):
    ''' 
    Runs a simple benchmark model which compares counts against an empirical CDF of observed historical counts in that bin and the adjacent bins
    window_radius_fb - number of adjacent fine bins to used for the comparison
    historical_weeks_used - the number of historical weeks used for the comparison
    '''
    # Get users to loop over and init output
    n_users = intract_tbl_frst_int.shape[0]
    output = np.zeros((2,4), dtype=np.float64)
    
    for user_id in range(n_users):

        # Getting rows to iterate over in counts df for that user
        user_first_row = intract_tbl_frst_int[user_id]
        user_last_row = intract_tbl_lst_int[user_id]

        ## Building an array of user counts including 0 rows lenght is test end (fine bins in dataset)
        user_counts = np.zeros(test_end, dtype='int64')
        for row in range(user_first_row, user_last_row + 1):
            fine_bin_id = cnts_tbl_f_bn_id[row]
            if fine_bin_id < test_end:
                user_counts[fine_bin_id] = cnts_tbl_cnt[row]

        for fine_bin_id in range(validation_start, test_end):
            period_idx = get_time_period(fine_bin_id, validation_start, validation_end, test_start, test_end)

            if period_idx == -1:
                continue

            # Getting the count for comparison and initalising outputs
            comparison_count = user_counts[fine_bin_id]
            observed_greater_or_equal_to_count = 0
            number_of_bins = 0
            observed_equal_to_count = 0

            # looping over historical fine bins and finding the historical observed count using our array
            for weeks_ago  in range(1, historical_weeks_used + 1):
                central_fine_bin = fine_bin_id - weeks_ago * fine_bins_per_week
                for fine_bin_offset in range(-window_radius_fb, window_radius_fb + 1):
                    historical_fine_bin = central_fine_bin + fine_bin_offset
                    assert historical_fine_bin >= 0
                    historical_observed_count = user_counts[historical_fine_bin]

                    # Calculating the upper tail probability and updating the output metrics
                    number_of_bins += 1
                    if historical_observed_count >= comparison_count:
                        observed_greater_or_equal_to_count +=1
                    if historical_observed_count == comparison_count:
                        observed_equal_to_count += 1

            upper_tail_prob = observed_greater_or_equal_to_count / number_of_bins
            strict_upper_tail_prob = (observed_greater_or_equal_to_count - observed_equal_to_count) / number_of_bins

            output[period_idx, 0] += observed_greater_or_equal_to_count
            output[period_idx, 1] += upper_tail_prob
            output[period_idx, 2] += strict_upper_tail_prob
            output[period_idx, 3] += 1

    return output

In [29]:
results

[{'name': 'raw_model',
  'w': 0.1,
  'cluster_param': 0,
  'smoothing_strength': 0,
  'validation_size': np.float64(45190656.0),
  'validation_log_likelihood': np.float64(-182375526.38562024),
  'validation_mean_log_likelihood': np.float64(-4.0356910593557265)},
 {'name': 'raw_model',
  'w': 0.01,
  'cluster_param': 0,
  'smoothing_strength': 0,
  'validation_size': np.float64(45190656.0),
  'validation_log_likelihood': np.float64(-203266276.1256366),
  'validation_mean_log_likelihood': np.float64(-4.497971353317744)}]